# Citrus: A real-world LSA example

Here is a real-world example of Land Suitability Analysis (LSA) using citrus and the North Island of New Zealand as study case. The goal of this notebook is to illustrate how to setup and perform a full LSA using LSAPy.

## Criteria

For this example, the criteria used to evaluate citrus land suitability are based on the [NZLUSDB project](https://baptistehamon.github.io/nzlusdb/) (NZ Land Use Suitability Database). The criteria are derived from the rules defined in the [DataSupermarket dataset](https://landuseopportunities.nz/dataset/citrus-crop-suitability-maps-rules-and-yield-information). There are a total of 12 criteria divided into two main categories: soil/terrain and climate. The criteria, their categories, and the corresponding indicators are listed in the table below:

| Criteria | Category | Indicator |
|----------|----------| --------- |
| Soil Drainage | Soil/Terrain | Soil Drainage Class |
| Soil pH | Soil/Terrain | Soil pH |
| Soil Rooting Depth | Soil/Terrain | Potential Rooting Depth (cm) |
| Soil Plant Available Water | Soil/Terrain | Soil Potential Plant Available Water (mm) |
| Soil Salinity | Soil/Terrain | Salinity |
| Soil Slope | Soil/Terrain | Slope (degrees) |
| Soil Stone Content | Soil/Terrain | Topsoil Gravel Content (%) |
| Rainfall Excess | Climate | Annual total precipitation (mm) |
| Flowering Mean Temperature | Climate | Mean daily temperature between Sep 15 and Nov 15 (°C) |
| Budburst Mean Minimum Temperature | Climate | Mean daily minimum temperature between Aug 15 and Oct 15 (°C) |
| Fruit Growth Mean Maximum Temperature | Climate | Mean daily maximum temperature between Jan 1 and Feb 15 (°C) |
| Harvest Heat Stress | Climate | Number of years with at least one hot week (3 days over 35C in a 7-day period) between Dec 1 and Feb 28 (over 10 years) |

```{note}
1. The periods used for climate indicators correspond to South Hemisphere growing seasons. Conducting the same analysis in the North Hemisphere would require to shift the periods by 6 months.
2. The NZLUSDB computes the Net Irrigation Requirements (NIR) alongside the suitability, and then assumes that the crop water needs are satisfied. As a result, no crop water requirement criterion is used in the NZLUSDB citrus LSA, as is the case in this example.
```

## Indicators

All the indicators used in this case study are available in the [lsapy-data](https://github.com/baptistehamon/lsapy-data) repository. You can use the `lsapy.tutorial` module to access the data.

### Opening the data

In [ ]:
import matplotlib.pyplot as plt

import lsapy
import lsapy.standardize as lstd
from lsapy import LandSuitabilityAnalysis, SuitabilityCriteria

data = lsapy.tutorial.open_dataset("citrus-dataset")
data

All the criteria indicators have `lon` and `lat` as spatial dimensions, and the climate-related indicators have an additional `time` dimension. In this example, the `time` dimension corresponds to annual values spanning 5 years (2000-2004).

### Visualizing the indicators

We can quickly plot the indicators to see what the data looks like.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 15))
for i, ax in enumerate(axes.flatten()):
    if i < len(data.data_vars):
        ind = list(data.data_vars)[i]
        # If the variable is a climate variable (i.e., 3D with a time dimension),
        # we have to select only the first time step to have 2D data to plot.
        kwargs = {"time": 0} if "time" in data[ind].dims else {}
        data[ind].isel(kwargs).plot(ax=ax)
        ax.set_title(ind)
    else:
        ax.axis("off")
plt.tight_layout()

## Setting up the LSA

### Defining the criteria

The first thing to do is to define the criteria that will be used in the LSA. Each criteria is defined by its name, category, and a weight giving its relative importance compared to the other criteria. Moreover, the indicator and the standardization function and its parameters used to evaluate the suitability of each criteria is also defined. The function and parameters used in this example are those defined in the [NZLUSDB project](https://baptistehamon.github.io/nzlusdb/).

In [ ]:
sc = {
    "drainage_class": SuitabilityCriteria(
        name="drainage_class",
        long_name="Soil Drainage Class",
        indicator=data["drainage"],
        weight=1,
        category="soilTerrain",
        func=lstd.discrete,
        fparams={"rules": {0: 0, 1: 0.1, 2: 0.6, 3: 0.9, 4: 1, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}},
    ),
    "ph": SuitabilityCriteria(
        name="ph",
        long_name="Soil pH",
        indicator=data["ph"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2024_eq10,
        fparams={"a": 0.0003683, "b": 5.974, "c": 2.767},
    ),
    "potential_rooting_depth": SuitabilityCriteria(
        name="potential_rooting_depth",
        long_name="Potential Rooting Depth",
        indicator=data["potential_rooting_depth"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -10.27, "b": 0.5506},
    ),
    "plant_available_water": SuitabilityCriteria(
        name="plant_available_water",
        long_name="Soil Potential Plant Available Water (mm)",
        indicator=data["profile_total_available_water"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -0.6951, "b": 150.1},
    ),
    "salinity": SuitabilityCriteria(
        name="salinity",
        long_name="Salinity",
        indicator=data["salinity"],
        weight=1,
        category="soilTerrain",
        func=lstd.boolean,
        fparams={"op": "<", "thresh": 0.1},
    ),
    "slope": SuitabilityCriteria(
        name="slope",
        long_name="Slope",
        indicator=data["slope"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": 3.629, "b": 10.70},
    ),
    "stone_content": SuitabilityCriteria(
        name="stone_content",
        long_name="Topsoil Stone Content",
        indicator=data["topsoil_gravel_content"],
        weight=1,
        category="soilTerrain",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": 0.9353, "b": 39.34},
    ),
    "rainfall_excess": SuitabilityCriteria(
        name="rainfall_excess",
        long_name="Rainfall excess",
        indicator=data["prcptot"],
        weight=1,
        category="climate",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": 0.5805, "b": 2080},
    ),
    "flowering_temperature": SuitabilityCriteria(
        name="flowering_temperature",
        long_name="Mean temperature during flowering (Sep 15 to Nov 15)",
        indicator=data["tg_mean"],
        weight=1.5,
        category="climate",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -15.14, "b": 10.85},
    ),
    "budburst_minimum_temperature": SuitabilityCriteria(
        name="budburst_minimum_temperature",
        long_name="Mean daily minimum temperature during budburst (Aug 1 to Sep 15)",
        indicator=data["tn_mean"],
        weight=1.5,
        category="climate",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -7.315, "b": 2.84},
    ),
    "fruit_growth_maximum_temperature": SuitabilityCriteria(
        name="fruit_growth_maximum_temperature",
        long_name="Mean daily maximum temperature during fruit growth (Nov 15 to Jan 15)",
        indicator=data["tx_mean"],
        weight=1.5,
        category="climate",
        func=lstd.vetharaniam2022_eq5,
        fparams={"a": -5.576, "b": 14.55},
    ),
    "harvest_heat_stress": SuitabilityCriteria(
        name="harvest_heat_stress",
        long_name=(
            "Number of years with at least one hot week (3 days over 35C in a 7-day period)"
            " between Dec 1 and Feb 28 (over 10 years)"
        ),
        indicator=data["year_with_hot_week"],
        weight=1.5,
        category="climate",
        func=lstd.vetharaniam2022_eq3,
        fparams={"a": -2.326, "b": 2.143},
    ),
}

In this example, a continuous standardization function is used for most of the criteria, but for the soil drainage and salinity criteria, a discrete and a boolean functions are used, respectively. The standardization functions and their parameters should be defined according to the land use and criteria being evaluated. Therefore, it is important to refer to the relevant literature or expert knowledge when defining the standardization functions and their parameters.

```{note}
For soil drainage, only the five first classes are relevant (i.e., _very poor_, _poor_, _imperfect_, _moderately well_ and _well_ drained soils), and a value of 0 is assigned to all the other classes (5 to 12) that correspond to other types of soil without drainage information (e.g., bare rock, lake, town, quarry, etc).
```

### Defining the LSA

Then, we can define the LSA using the `LandSuitabilityAnalysis` class. The LSA is defined by its name and the criteria used.

In [ ]:
lsa = LandSuitabilityAnalysis(
    land_use="citrus", short_name="citrus_suitability", long_name="Citrus Suitability", criteria=sc
)

## Computing citrus land suitability

```{warning}
While this example is based on scientific literature and expert knowledge, the absence of NIR assessment does not allow to fully reproduce the assumptions made in the NZLUSDB citrus LSA. Therefore, the results of this analysis should not be used for any decision-making purposes, and the results are only presented here for tutorial purposes.
```

### Criteria suitability

Once the LSA is defined, the suitability can be computing using the `run` method. Here, we set `suitability_type="criteria"` to only trigger the computation of the the criteria, and then plot the results.

In [ ]:
res = lsa.run(suitability_type="criteria")
res = res[[v for v in sc.keys()]]  # just to ensure same order for plotting
res

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 15))
for i, ax in enumerate(axes.flatten()):
    if i < len(res.data_vars):
        var = list(res.data_vars)[i]
        # If the variable is a climate variable (i.e., 3D with a time dimension),
        # we have to select only the first time step to have 2D data to plot.
        kwargs = {"time": 0} if "time" in res[var].dims else {}
        res[var].isel(kwargs).plot(ax=ax, vmin=0, vmax=1)
        ax.set_title(var)
    else:
        ax.axis("off")
plt.tight_layout()

Different suitability patterns are observed depending on the criteria. For example, the salinity, stone content, maximum temperature during fruit growth, and heat stress criteria are almost fully suitable for all the North Island, while the plant available water seems to be the main limiting factor for citrus growth.

### Category and overall suitability

By setting `suitability_type="overall"`, the `run` method will by default compute the suitability for each category if some exist, and then compute the overall suitability from the category suitability. In this example, the weighted geometric mean is used to compute the suitability by setting `agg_methods="wgmean"` (see the [aggregation methods](../user-guide/aggregation.md) user guide for more details).

In [ ]:
res = lsa.run(suitability_type="overall", agg_methods="wgmean")  # compute category and overall suitability
res = res[[v for v in sc.keys()] + ["soilTerrain", "climate", "suitability"]]  # reorder
res

By default, the suitability of all criteria and categories is returned along with the overall suitability; this can be controlled using the `keep_vars` argument.

Below we plot the soil/terrain, climate and overall suitability.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
res["soilTerrain"].plot(ax=ax[0], vmin=0, vmax=1)
ax[0].set_title("Soil and Terrain Suitability")
res["climate"].isel(time=0).plot(ax=ax[1], vmin=0, vmax=1)
ax[1].set_title("Climate Suitability")
res["suitability"].isel(time=0).plot(ax=ax[2], vmin=0, vmax=1)
ax[2].set_title("Overall Suitability")
plt.tight_layout()
plt.show()

We can observe that the suitability is mainly limited by the soil/terrain criteria, and that the climate criteria are almost fully suitable for all the North Island. The citrus land suitability is high for only a limited area, with the majority of the North Island being moderately suitable and with small unsuitable areas.

### Temporal suitability

When computing the climate and overall suitability, LSAPy automatically [broadcasts](https://docs.xarray.dev/en/latest/user-guide/terminology.html#term-Broadcasting) arrays with different dimensions. In this case, the climate and overall suitability are computed for each year, resulting in suitability arrays with a `time` dimension corresponding to the 5 years of data. We can thus explore how the suitability changes over time. Below, we compute the mean suitability for over the 5 years and plot the difference for the given years.

In [ ]:
mean = res["suitability"].mean(dim="time")
changes = res["suitability"] - mean
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
mean.plot(ax=axes[0], vmin=0, vmax=1)
axes[0].set_title("2000-2005 Mean")
for i, ax, t in zip(range(1, 4), axes[1:], [2000, 2002, 2005], strict=True):
    changes.isel(time=i - 1).plot(ax=ax, vmin=-0.15, vmax=0.15)
    ax.set_title(f"{t} vs 2000-2005")
plt.tight_layout()

The suitability is relatively stable over the 3 years selected, with only small differences observed. This is not surprising, as (i) a continuous 5-year period is used with a limited interannual climate variability, and (ii) climate criteria are fully suitable for most of the North Island, and thus do not limit overall suitability. However, the suitability is expected to be more variable over longer periods of time or a different time horizon.

## Statistic summaries

It is often useful to have summary information about analysis. LSAPy provides the `stats_summary` and `spatial_stats_summary` functions to compute statistics summaries from the LSA results. The `stats_summary` function computes descriptive statistics for each suitability variable along spatial dimensions, while `spatial_stats_summary` does the same but for different spatial areas provided as a `GeoDataFrame`. Below we compute the statistics summaries only for the overall suitability for simplicity, but the same can be done for all the criteria and category suitability variables.

## Basic summary

In [ ]:
from lsapy.stats import spatial_stats_summary, stats_summary

stats = stats_summary(res, on_vars=["suitability"], dropna=True)
stats

The result is a `pandas.DataFrame` with columns corresponding to descriptive statistics. All non-spatial dimensions are also kept; in this case, the `time` dimension, and thus the statistics are computed for each year.

### Spatial summary

For the spatial summary, the administrative regions of New Zealand are used as spatial areas, and they can be accessed using the `lsapy.tutorial.open_dataset` function with `nz-regions` as dataset name.

In [ ]:
regions = lsapy.tutorial.open_dataset("nz-regions")
regions = regions.to_crs(epsg=4326)  # convert to the same CRS than indicators
regions

We can quickly visualize the regions:

In [ ]:
labels = [r.replace(" Region", "") for r in regions["REGC2025_1"].values]

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
regions.plot(ax=ax, column="REGC2025_V", legend=True, legend_kwds={"labels": labels})
plt.tight_layout()

In [ ]:
stats = spatial_stats_summary(
    res,
    regions,
    name="region",
    mask_kwargs={"names": "REGC2025_1"},
    on_vars=["suitability"],
    dropna=True,
)
# print results for Auckland and Wellington Region only
stats[stats["region"].isin(["Auckland", "Wellington Region"])]